In [ ]:
import numpy as np
import pandas as pd
import math
from collections import defaultdict

In [ ]:
def read_conllu(file_path):
    # Initialisation d'une liste pour stocker toutes les phrases
    sentences = []

    # Ouverture du fichier .conllu en mode lecture avec encodage UTF-8
    with open(file_path, 'r', encoding='utf-8') as f:
        sentence = []  # Liste temporaire pour stocker les mots d'une phrase

        # Parcours ligne par ligne du fichier
        for line in f:
            line = line.strip()  # Suppression des espaces et sauts de ligne

            # Si la ligne est vide, cela indique la fin d'une phrase
            if not line:
                if sentence:
                    sentences.append(sentence)  # On ajoute la phrase complète
                    sentence = []  # On réinitialise la phrase

            # Si la ligne n'est pas un commentaire (ne commence pas par '#')
            elif not line.startswith("#"):
                parts = line.split('\t')  # Découpe la ligne en colonnes

                # Vérifie qu'il y a au moins 4 colonnes et que ce n'est pas une ligne de token "multi-mot" (ex: 2-3)
                if len(parts) > 3 and '-' not in parts[0]:
                    word = parts[1]  # Le mot (forme de surface)
                    upos = parts[3]  # L'étiquette universelle (Universal POS tag)
                    sentence.append((word, upos))  # Ajoute le tuple (mot, étiquette) à la phrase

        # Si une phrase est encore en mémoire après la dernière ligne, on l'ajoute
        if sentence:
            sentences.append(sentence)

    # Retourne la liste de phrases, chaque phrase étant une liste de tuples (mot, étiquette)
    return sentences


In [ ]:
cont=read_conllu("ANTILLES/train.conllu")

In [ ]:
def train_hmm(sentences):
    # Dictionnaire de transitions : transition[tag_précédent][tag_actuel] = fréquence
    transition = defaultdict(lambda: defaultdict(int))

    # Dictionnaire d’émissions : emission[tag][mot] = fréquence
    emission = defaultdict(lambda: defaultdict(int))

    # Compteur du nombre total d’occurrences de chaque tag
    tag_counts = defaultdict(int)

    # Compteur des tags de début de phrase (utile pour les probabilités initiales si besoin)
    start_counts = defaultdict(int)

    # Boucle sur chaque phrase
    for sentence in sentences:
        prev_tag = "<START>"  # Tag de départ fictif pour le début de chaque phrase

        # Parcours mot par mot la phrase
        for word, tag in sentence:
            transition[prev_tag][tag] += 1  # Incrémente la transition entre le tag précédent et le tag actuel
            emission[tag][word] += 1        # Incrémente le compteur d’émission du mot par ce tag
            tag_counts[tag] += 1            # Incrémente le nombre total d’occurrences de ce tag
            prev_tag = tag                  # Le tag actuel devient le précédent pour l’itération suivante

        transition[prev_tag]["<END>"] += 1  # On marque la fin de phrase (dernier tag -> <END>)
        start_counts[sentence[0][1]] += 1   # Incrémente le tag qui commence cette phrase

    # Normalisation : calcul des probabilités de transition (en log-proba)
    transition_probs = defaultdict(dict)
    for prev_tag in transition:
        total = sum(transition[prev_tag].values())  # Total des transitions depuis ce tag

        for tag in transition[prev_tag]:
            # Probabilité log (plus stable numériquement que les probas brutes)
            transition_probs[prev_tag][tag] = math.log(transition[prev_tag][tag] / total)

    # Normalisation : calcul des probabilités d’émission (en log-proba)
    emission_probs = defaultdict(dict)
    for tag in emission:
        total = sum(emission[tag].values())  # Total des mots émis par ce tag

        for word in emission[tag]:
            emission_probs[tag][word] = math.log(emission[tag][word] / total)

    # Retourne les probabilités de transition, d’émission, et l’ensemble des tags rencontrés
    return transition_probs, emission_probs, set(tag_counts)

In [ ]:
a,b,c=train_hmm(cont)


In [ ]:
def viterbi(sentence, transition_probs, emission_probs, tags):
    V = [{}]       # Matrice Viterbi : V[t][tag] = log-proba max pour tag à la position t
    path = {}      # Dictionnaire des chemins : path[tag] = meilleure séquence de tags menant à `tag` à t=0

    # Initialisation pour le premier mot de la phrase
    for tag in tags:
        # Probabilité de transition depuis <START> vers ce tag
        transition_log = transition_probs["<START>"].get(tag, float('-inf'))  # -inf si transition absente

        # Probabilité d’émission du premier mot par ce tag (avec lissage)
        emission_log = emission_probs[tag].get(sentence[0], math.log(1e-6))

        # Calcul de la log-proba totale pour ce tag au temps t=0
        V[0][tag] = transition_log + emission_log

        # On initialise le chemin vers chaque tag par ce tag lui-même
        path[tag] = [tag]

    # Parcours des mots de la phrase à partir du second (t=1)
    for t in range(1, len(sentence)):
        V.append({})         # Ajoute un nouveau niveau dans la matrice Viterbi
        new_path = {}        # Nouveau dictionnaire de chemins à l'étape t

        for curr_tag in tags:
            # Pour chaque tag courant, on cherche la meilleure transition depuis un tag précédent
            max_prob, best_prev_tag = max(
                (
                    V[t-1][prev_tag] +                                        # log-proba jusqu'à prev_tag
                    transition_probs[prev_tag].get(curr_tag, math.log(1e-6)) +  # log-proba de transition vers curr_tag
                    emission_probs[curr_tag].get(sentence[t], math.log(1e-6)),  # log-proba d’émission du mot courant
                    prev_tag
                )
                for prev_tag in tags if prev_tag in V[t-1]  # On ne considère que les tags valides à t-1
            )

            # Mise à jour de la matrice Viterbi avec la meilleure probabilité
            V[t][curr_tag] = max_prob

            # Mise à jour du chemin : on ajoute le tag courant au meilleur chemin trouvé
            new_path[curr_tag] = path[best_prev_tag] + [curr_tag]

        # Mise à jour des chemins pour l’étape suivante
        path = new_path

    # Fin de phrase : on cherche le tag final qui mène à la proba maximale en tenant compte de la transition vers <END>
    max_prob, best_last_tag = max(
        (V[-1][tag] + transition_probs[tag].get("<END>", 0), tag)
        for tag in tags
    )

    # Retourne la meilleure séquence de tags pour la phrase
    return path[best_last_tag]


In [ ]:
# Exemple d'utilisation :

# 1. Lecture des données d'entraînement au format CoNLL-U
sentences = read_conllu("ANTILLES/train.conllu")
# Cette fonction lit le fichier 'train.conllu' et retourne une liste de phrases,
# où chaque phrase est une liste de tuples (mot, étiquette grammaticale)

# 2. Entraînement du modèle HMM
trans_probs, emis_probs, all_tags = train_hmm(sentences)
# - trans_probs : probabilités de transition entre les étiquettes
# - emis_probs : probabilités d’émission d’un mot donné par une étiquette
# - all_tags : ensemble de toutes les étiquettes rencontrées

# 3. Phrase de test (séquence de mots sans étiquettes)
test_sentence = ["Le", "chat", "mange", "une", "souris"]

# 4. Prédiction des étiquettes avec l’algorithme de Viterbi
predicted_tags = viterbi(test_sentence, trans_probs, emis_probs, all_tags)
# Cette fonction retourne la meilleure séquence de tags (étiquettes grammaticales)
# pour la phrase testée, basée sur le modèle HMM appris

# 5. Affichage du résultat : chaque mot est affiché avec son étiquette prédite
print(list(zip(test_sentence, predicted_tags)))
# Exemple de sortie : [('Le', 'DET'), ('chat', 'NOUN'), ('mange', 'VERB'), ...]


[('Le', 'DETMS'), ('chat', 'NMS'), ('mange', 'VERB'), ('une', 'DINTFS'), ('souris', 'NFS')]


In [ ]:
sentences2 = read_conllu("ANTILLES/test.conllu")

In [ ]:
def evaluate_viterbi_model(filepath, known_tags, transition_probs, emission_probs):
    # Lecture du fichier CoNLL-U de test : retourne une liste de phrases
    # Chaque phrase est une liste de tuples (mot, étiquette)
    sentences = read_conllu(filepath)

    # Listes pour stocker toutes les étiquettes réelles et prédites
    all_gold_tags = []        # étiquettes de référence (gold standard)
    all_predicted_tags = []   # étiquettes prédites par le modèle

    # Parcours de chaque phrase du fichier
    for sentence in sentences:
        # Extraction des mots de la phrase (sans les tags)
        words = [word for word, tag in sentence]

        # Extraction des étiquettes correctes de la phrase
        gold_tags = [tag for word, tag in sentence]

        # Prédiction des étiquettes à l'aide de Viterbi
        predicted_tags = viterbi(words, transition_probs, emission_probs, known_tags)

        # Ajout des résultats à l’ensemble global
        all_gold_tags.extend(gold_tags)
        all_predicted_tags.extend(predicted_tags)

    # Calcul de la précision globale (accuracy) entre vraies et prédictions
    acc = accuracy_score(all_gold_tags, all_predicted_tags)

    # Affichage du score avec 4 décimales
    print(f"Accuracy: {acc:.4f}")

    # Retourne la précision pour éventuellement la réutiliser
    return acc


In [ ]:
from sklearn.metrics import accuracy_score
accuracy=evaluate_viterbi_model("ANTILLES/test.conllu", all_tags, trans_probs, emis_probs)

Accuracy: 0.9218
